In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from src.models import whiff

wm = whiff.build()

eval_part = wm.split.validation.iloc[wm.n_calibration:].copy()
eval_part["y"] = eval_part["target"].to_numpy()
y_eval = eval_part["y"].to_numpy()

def score(p, name):
    return {
        "model": name,
        "log_loss": log_loss(y_eval, p),
        "brier": brier_score_loss(y_eval, p),
        "auc": roc_auc_score(y_eval, p),
        "ece": whiff_ece(y_eval, p),
    }

def whiff_ece(y, p, bins=10):
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    idx = np.digitize(p, edges[1:-1])
    total = 0.0
    for b in range(bins):
        m = idx == b
        if m.sum() == 0:
            continue
        total += m.sum() * abs(y[m].mean() - p[m].mean())
    return total / len(y)

results = [
    score(wm.predict_baseline(eval_part), "lookup baseline"),
    score(wm.predict(eval_part), "logistic D (threshold 500)"),
]
print(pd.DataFrame(results).round(5).to_string(index=False))

                     model  log_loss   brier     auc     ece
           lookup baseline   0.48102 0.15451 0.71205 0.01044
logistic D (threshold 500)   0.46449 0.14694 0.73090 0.00883


In [2]:
from src.models.whiff import (
    build_features, calibrate, fit_logistic, fit_lookup_baseline
)

def try_threshold(threshold):
    counts = wm.split.train["pitch_type"].value_counts()
    keep = set(counts[counts >= threshold].index)

    X_tr = build_features(wm.split.train, keep)
    X_va = build_features(wm.split.validation, keep, columns=X_tr.columns)
    m = fit_logistic(X_tr, wm.y_train)

    n_cal = wm.n_calibration
    cal = calibrate(m, X_va.iloc[:n_cal], wm.y_validation[:n_cal])
    p = cal.predict_proba(X_va.iloc[n_cal:])[:, 1]
    return keep, p, X_tr.shape[1]

for th in [500, 1500, 3000, 6000]:
    keep, p, n_feat = try_threshold(th)
    r = score(p, f"logistic (threshold {th})")
    r["kept_types"] = len(keep)
    r["features"] = n_feat
    results.append(r)

print(pd.DataFrame(results).round(5).to_string(index=False))

                     model  log_loss   brier     auc     ece  kept_types  features
           lookup baseline   0.48102 0.15451 0.71205 0.01044         NaN       NaN
logistic D (threshold 500)   0.46449 0.14694 0.73090 0.00883         NaN       NaN
  logistic (threshold 500)   0.46449 0.14694 0.73090 0.00883        10.0      31.0
 logistic (threshold 1500)   0.46454 0.14695 0.73085 0.00971         9.0      29.0
 logistic (threshold 3000)   0.46454 0.14695 0.73085 0.00971         9.0      29.0
 logistic (threshold 6000)   0.46447 0.14693 0.73080 0.00979         8.0      27.0


In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier

# HistGradientBoosting handles NaN natively and is fast on 200k rows.
# No scaling needed — trees are invariant to monotonic transforms.
X_tr = wm.X_train
X_va = wm.X_validation
n_cal = wm.n_calibration

gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,
)
gb.fit(X_tr, wm.y_train)
print("iterations used:", gb.n_iter_)

gb_cal = calibrate(gb, X_va.iloc[:n_cal], wm.y_validation[:n_cal])
p_gb_raw = gb.predict_proba(X_va.iloc[n_cal:])[:, 1]
p_gb_cal = gb_cal.predict_proba(X_va.iloc[n_cal:])[:, 1]

results.append(score(p_gb_raw, "boosting (uncalibrated)"))
results.append(score(p_gb_cal, "boosting (isotonic)"))

print(pd.DataFrame(results).round(5).to_string(index=False))

iterations used: 292
                     model  log_loss   brier     auc     ece  kept_types  features
           lookup baseline   0.48102 0.15451 0.71205 0.01044         NaN       NaN
logistic D (threshold 500)   0.46449 0.14694 0.73090 0.00883         NaN       NaN
  logistic (threshold 500)   0.46449 0.14694 0.73090 0.00883        10.0      31.0
 logistic (threshold 1500)   0.46454 0.14695 0.73085 0.00971         9.0      29.0
 logistic (threshold 3000)   0.46454 0.14695 0.73085 0.00971         9.0      29.0
 logistic (threshold 6000)   0.46447 0.14693 0.73080 0.00979         8.0      27.0
   boosting (uncalibrated)   0.43472 0.13719 0.77855 0.00575         NaN       NaN
       boosting (isotonic)   0.43981 0.13732 0.77834 0.00667         NaN       NaN


In [4]:
eval_part["p_gb"] = p_gb_cal
eval_part["p_log"] = wm.predict(eval_part)
eval_part["p_base"] = wm.predict_baseline(eval_part)

pt = eval_part["pitch_type"].astype(str)
eval_part["pitch_type_b"] = pt.where(pt.isin(wm.keep_types), "OTHER")

rows = []
for key, g in eval_part.groupby("pitch_type_b"):
    if len(g) < 500:
        continue
    rows.append({
        "pitch": key, "n": len(g),
        "baseline": log_loss(g["y"], g["p_base"], labels=[0, 1]),
        "logistic": log_loss(g["y"], g["p_log"], labels=[0, 1]),
        "boosting": log_loss(g["y"], g["p_gb"], labels=[0, 1]),
    })
by_pt = pd.DataFrame(rows)
by_pt["gb_vs_base"] = by_pt["baseline"] - by_pt["boosting"]
by_pt["gb_vs_log"] = by_pt["logistic"] - by_pt["boosting"]
print(by_pt.sort_values("gb_vs_base").round(4).to_string(index=False))

pitch     n  baseline  logistic  boosting  gb_vs_base  gb_vs_log
   KC   769    0.5457    0.5782    0.5498     -0.0041     0.0284
   SI  5966    0.3275    0.3352    0.3109      0.0166     0.0243
   FF 13391    0.4564    0.4367    0.4257      0.0307     0.0110
   CH  4735    0.5787    0.5438    0.5343      0.0444     0.0095
   FC  3661    0.4639    0.4888    0.4186      0.0452     0.0702
   ST  2973    0.5384    0.5171    0.4928      0.0456     0.0243
   SL  5990    0.5392    0.5146    0.4741      0.0651     0.0405
   FS  1464    0.5752    0.5312    0.5082      0.0670     0.0231
   CU  2428    0.5399    0.4856    0.4597      0.0801     0.0258


In [6]:
from sklearn.inspection import permutation_importance

# Sample by POSITION, not label. X_va carries the original frame's index,
# so label arithmetic against a numpy array does not line up.
X_eval_full = X_va.iloc[n_cal:]
y_eval_full = wm.y_validation[n_cal:]

rng = np.random.default_rng(42)
pos = rng.choice(len(X_eval_full), size=8000, replace=False)

imp = permutation_importance(
    gb, X_eval_full.iloc[pos], y_eval_full[pos],
    scoring="neg_log_loss", n_repeats=5, random_state=42, n_jobs=-1,
)
top = pd.Series(imp.importances_mean, index=X_eval_full.columns).sort_values(ascending=False)
print(top.head(12).round(5).to_string())

plate_z_rel             0.14471
plate_x_bat             0.04452
strikes                 0.01832
pitch_type_b_FF         0.00620
release_speed           0.00565
pfx_z                   0.00475
pitch_type_b_SI         0.00383
release_spin_rate       0.00319
pfx_x                   0.00245
stand_R                 0.00196
p_throws_R              0.00127
pitch_type_b_SI_x_pz    0.00078


In [7]:
configs = [
    {"max_leaf_nodes": 15, "learning_rate": 0.06},
    {"max_leaf_nodes": 31, "learning_rate": 0.06},
    {"max_leaf_nodes": 63, "learning_rate": 0.06},
    {"max_leaf_nodes": 31, "learning_rate": 0.03},
]
for cfg in configs:
    m = HistGradientBoostingClassifier(
        max_iter=500, l2_regularization=1.0, early_stopping=True,
        validation_fraction=0.15, random_state=42, **cfg)
    m.fit(X_tr, wm.y_train)
    p = m.predict_proba(X_va.iloc[n_cal:])[:, 1]
    print(f"{cfg}  iters={m.n_iter_:3d}  log_loss={log_loss(y_eval, p):.5f}  "
          f"ece={whiff_ece(y_eval, p):.5f}")

{'max_leaf_nodes': 15, 'learning_rate': 0.06}  iters=322  log_loss=0.43594  ece=0.00694
{'max_leaf_nodes': 31, 'learning_rate': 0.06}  iters=292  log_loss=0.43472  ece=0.00575
{'max_leaf_nodes': 63, 'learning_rate': 0.06}  iters=161  log_loss=0.43535  ece=0.00618
{'max_leaf_nodes': 31, 'learning_rate': 0.03}  iters=424  log_loss=0.43523  ece=0.00665
